In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import shap
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler

project_root = Path.cwd().parent
data_dir = project_root / 'data'
models_dir = project_root / 'models'
reports_dir = project_root / 'reports'
models_dir.mkdir(exist_ok=True)
reports_dir.mkdir(exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.2
NUMERIC_FEATURES = [
    "query_word_count", "query_avg_word_length", "query_has_question_mark",
    "query_number_density", "query_capitalized_density",
    "reasoning_step_count", "ambiguity_score",
]
CATEGORICAL_FEATURES = ["domain", "question_type", "source_category"]

df = pd.read_parquet(data_dir / "features.parquet")
encoded = pd.get_dummies(df[CATEGORICAL_FEATURES], prefix=CATEGORICAL_FEATURES)
X = pd.concat([df[NUMERIC_FEATURES], encoded], axis=1)
y = df["label"]
print(f"Loaded {len(X)} rows, {len(X.columns)} encoded features. label counts: {y.value_counts().to_dict()}")

In [ ]:
# Stratified split -- with only 68 label=0 rows total, an unstratified split risks a
# very unbalanced test fold by chance.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y,
)
print(f"Train: {len(X_train)} rows ({(y_train == 0).sum()} label=0, {(y_train == 1).sum()} label=1)")
print(f"Test:  {len(X_test)} rows ({(y_test == 0).sum()} label=0, {(y_test == 1).sum()} label=1)")

In [ ]:
def train_logistic_regression(X_train, y_train):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    model = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)
    model.fit(X_train_scaled, y_train)
    return model, scaler

def train_xgboost(X_train, y_train):
    # Regularized deliberately: only 54-68 label=0 examples total, so XGBoost's default
    # flexibility (deep trees, no subsampling) overfits badly on this minority class.
    scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
    model = xgb.XGBClassifier(
        scale_pos_weight=scale_pos_weight,
        max_depth=3, min_child_weight=3, subsample=0.8, colsample_bytree=0.8,
        reg_lambda=2.0, n_estimators=100, random_state=RANDOM_STATE, eval_metric="logloss",
    )
    model.fit(X_train, y_train)
    return model

def evaluate(model_name, y_true, y_pred, y_proba):
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    return {
        "model": model_name, "roc_auc": round(float(roc_auc_score(y_true, y_proba)), 4),
        "precision_label0": round(report["0"]["precision"], 4), "recall_label0": round(report["0"]["recall"], 4),
        "f1_label0": round(report["0"]["f1-score"], 4),
        "precision_label1": round(report["1"]["precision"], 4), "recall_label1": round(report["1"]["recall"], 4),
        "f1_label1": round(report["1"]["f1-score"], 4), "accuracy": round(report["accuracy"], 4),
        "support_label0": int(report["0"]["support"]), "support_label1": int(report["1"]["support"]),
    }

lr_model, scaler = train_logistic_regression(X_train, y_train)
X_test_scaled = scaler.transform(X_test)
lr_eval = evaluate("logistic_regression", y_test, lr_model.predict(X_test_scaled), lr_model.predict_proba(X_test_scaled)[:, 1])

xgb_model = train_xgboost(X_train, y_train)
xgb_eval = evaluate("xgboost", y_test, xgb_model.predict(X_test), xgb_model.predict_proba(X_test)[:, 1])

print(json.dumps({"logistic_regression": lr_eval, "xgboost": xgb_eval}, indent=2))

In [ ]:
# 5-fold CV -- the single held-out test set above has only ~14 label=0 examples, too few
# to trust a single split's ROC-AUC on its own. This is what caught XGBoost's single-split
# score (0.51, near-random) as an unlucky fold rather than the model's real performance.
def cross_validate(X, y, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    lr_aucs, xgb_aucs = [], []
    for train_idx, test_idx in skf.split(X, y):
        X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
        y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
        lr_m, sc = train_logistic_regression(X_tr, y_tr)
        lr_aucs.append(roc_auc_score(y_te, lr_m.predict_proba(sc.transform(X_te))[:, 1]))
        xgb_m = train_xgboost(X_tr, y_tr)
        xgb_aucs.append(roc_auc_score(y_te, xgb_m.predict_proba(X_te)[:, 1]))
    return {
        "n_splits": n_splits,
        "logistic_regression_roc_auc_mean": round(float(np.mean(lr_aucs)), 4),
        "logistic_regression_roc_auc_std": round(float(np.std(lr_aucs)), 4),
        "logistic_regression_roc_auc_per_fold": [round(float(a), 4) for a in lr_aucs],
        "xgboost_roc_auc_mean": round(float(np.mean(xgb_aucs)), 4),
        "xgboost_roc_auc_std": round(float(np.std(xgb_aucs)), 4),
        "xgboost_roc_auc_per_fold": [round(float(a), 4) for a in xgb_aucs],
    }

cv_results = cross_validate(X, y)
print(json.dumps(cv_results, indent=2))

eval_report = {"single_split": {"logistic_regression": lr_eval, "xgboost": xgb_eval}, "cross_validated": cv_results}
(reports_dir / "phase3_model_evaluation.json").write_text(json.dumps(eval_report, indent=2))

In [ ]:
# SHAP feature importance (XGBoost) -- which query features actually drive the routing decision
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)
mean_abs_shap = np.abs(shap_values).mean(axis=0)
shap_report = pd.DataFrame({"feature": X_test.columns, "mean_abs_shap": mean_abs_shap}).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
shap_report.to_csv(reports_dir / "phase3_shap_feature_importance.csv", index=False)
print(shap_report.to_string(index=False))

In [ ]:
# Save models
import joblib
xgb_model.save_model(models_dir / "xgboost_router.json")
joblib.dump({"model": lr_model, "scaler": scaler, "feature_names": list(X.columns)}, models_dir / "logistic_regression_router.pkl")
print("Saved models/xgboost_router.json and models/logistic_regression_router.pkl")